# Random Forest Prediction of EPVI Indicators

This notebook trains one random forest model per EPVI indicator from satellite-derived freguesia indicators and only the **basic** administrative indicators listed in `data/adm_data_split.json`.

The privately shared EPVI file stays outside git. Detailed administrative indicators remain excluded here; they are reserved for residual analysis.

Model selection now uses spatial folds built from `data/freguesias_to_NUTS3.csv`. The fixed NUTS3 holdout `PT16E` and `PT16J` is removed before tuning and is used only for final performance estimates.


In [1]:
from __future__ import annotations

import json
import sys
import time
from datetime import datetime
from importlib import import_module
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import ParameterGrid, RandomizedSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
try:
    from IPython.display import display
except Exception:
    display = print

RANDOM_STATE = 42


In [2]:
REPO_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pipeline" / "config").exists()
)
sys.path.append(str(REPO_ROOT))

from pipeline.utils.paths import load_paths, path_value, repo_data_path

rf_utils = import_module("pipeline.3_epvi_prediction.utils")
attach_nuts3 = rf_utils.attach_nuts3
build_modeling_table = rf_utils.build_modeling_table
choose_satellite_csv = rf_utils.choose_satellite_csv
id_coverage_summary = rf_utils.id_coverage_summary
load_nuts3_mapping = rf_utils.load_nuts3_mapping
load_prediction_inputs = rf_utils.load_prediction_inputs
make_nuts3_cv_splits = rf_utils.make_nuts3_cv_splits
rmse = rf_utils.rmse
spearman_corr = rf_utils.spearman_corr
split_fixed_nuts3_holdout = rf_utils.split_fixed_nuts3_holdout

PATHS = load_paths()
SAT_CSV, REGENERATED_SAT_CSV, SNAPSHOT_SAT_CSV = choose_satellite_csv(PATHS, repo_data_path, path_value)
ADM_CSV = repo_data_path(PATHS, "all_used_adm_indicators.csv")
ADM_SPLIT_JSON = repo_data_path(PATHS, "adm_data_split.json")
NUTS3_MAPPING_CSV = repo_data_path(PATHS, "freguesias_to_NUTS3.csv")
EPVI_CSV = path_value(PATHS, "epvi_csv")
MODEL_OUT_DIR = path_value(PATHS, "external_data_root") / "outputs" / "epvi_prediction" / "random_forest"
MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("repo:", REPO_ROOT)
print("satellite predictors:", SAT_CSV)
if SAT_CSV == SNAPSHOT_SAT_CSV:
    print("WARNING: regenerated external index CSV was not found; using committed satellite snapshot.")
print("administrative predictors:", ADM_CSV)
print("private EPVI targets:", EPVI_CSV)
print("NUTS3 mapping:", NUTS3_MAPPING_CSV)
print("model outputs:", MODEL_OUT_DIR)


repo: e:\git_projects\energy_poverty_from_space
satellite predictors: E:\OneDrive\Studia\Studia magisterskie\Masterarbeit 2 - Sozialwissenschaften\data\outputs\indices\freguesia_indices_streaming.csv
administrative predictors: E:\git_projects\energy_poverty_from_space\data\all_used_adm_indicators.csv
private EPVI targets: E:\OneDrive\Studia\Studia magisterskie\Masterarbeit 2 - Sozialwissenschaften\data\EPVI_results_gouveia_et_al_2019.csv
NUTS3 mapping: E:\git_projects\energy_poverty_from_space\data\freguesias_to_NUTS3.csv
model outputs: E:\OneDrive\Studia\Studia magisterskie\Masterarbeit 2 - Sozialwissenschaften\data\outputs\epvi_prediction\random_forest


In [3]:
inputs = load_prediction_inputs(
    sat_csv=SAT_CSV,
    adm_csv=ADM_CSV,
    epvi_csv=EPVI_CSV,
    adm_split_json=ADM_SPLIT_JSON,
)

EPVI_TARGETS = inputs["targets"]
model_df, predictor_cols, constant_cols = build_modeling_table(inputs)
model_df = attach_nuts3(model_df, load_nuts3_mapping(NUTS3_MAPPING_CSV))
train_df, test_df = split_fixed_nuts3_holdout(model_df)

print("Raw input ID coverage:")
display(id_coverage_summary(inputs))
print("targets:", EPVI_TARGETS)
print("Final modeling rows:", len(model_df))
print("Training rows outside fixed test NUTS3:", len(train_df))
print("Fixed test rows in PT16E/PT16J:", len(test_df))
print("Training NUTS3 regions:", train_df["NUTS3"].nunique())
print("Test NUTS3 regions:", sorted(test_df["NUTS3"].unique()))
print("Predictors used after dropping constant/all-missing columns:", len(predictor_cols))
print("Dropped constant/all-missing predictors:", constant_cols)
display(model_df[["ID_norm", "NUTS3", "parish_name", *EPVI_TARGETS]].head())
display(model_df[predictor_cols].isna().mean().sort_values(ascending=False).rename("missing_share").head(20).to_frame())


Raw input ID coverage:


,table,rows,unique_ids,duplicate_ids,missing_ids
0,sat,3092.0,3092,0.0,0.0
1,adm,3093.0,3092,0.0,1.0
2,epvi,3092.0,3092,0.0,0.0
3,common_sat_adm_epvi,NaN,3087,NaN,NaN


targets: ['EPG heating', 'EPG cooling', 'AIAM', 'EPVI heating', 'EPVI cooling']
Final modeling rows: 3087
Training rows outside fixed test NUTS3: 2653
Fixed test rows in PT16E/PT16J: 434
Training NUTS3 regions: 23
Test NUTS3 regions: ['PT16E', 'PT16J']
Predictors used after dropping constant/all-missing columns: 46
Dropped constant/all-missing predictors: ['res_vol_per_capita_p10', 'pop_density_p10', 'ntl_per_res_vol_p10', 'ntl_per_res_vol_p50', 'energy_per_vol_per_capita_p10']


,ID_norm,NUTS3,parish_name,EPG heating,EPG cooling,AIAM,EPVI heating,EPVI cooling
0,010103,PT16D,Aguada de Cima,15,19,13.4,10.8,12.8
1,010109,PT16D,Fermentelos,16,19,13.4,11.3,12.8
2,010112,PT16D,Macinhata do Vouga,16,19,12.4,11.8,13.3
3,010119,PT16D,Valongo do Vouga,16,19,13.8,11.1,12.6
4,010121,PT16D,União das Freguesias de Águeda e Borralha,14,20,13.1,10.5,13.5


,missing_share
extreme_cold,0.106252
extreme_heat,0.106252
cdd_25,0.106252
hdd_18,0.106252
res_ratio,0.017169
standard_delta_pm_25,0.017169
ntl_per_res_vol_p90,0.017169
ntl_per_res_vol_gini,0.017169
energy_per_vol_per_capita_p50,0.017169
energy_per_vol_per_capita_p90,0.017169


## Modeling Design

The first spatial tuning pass is broad again. Earlier random-row tuning used a focused parameter grid, but that choice is not assumed to transfer to region-held-out validation.

For each target, hyperparameters are searched only on training NUTS3 regions. Validation folds hold out whole NUTS3 regions and the fold builder keeps `PT200` and `PT300` in different folds when both are present. After tuning, the selected model is evaluated in two distinct ways:

- spatial out-of-fold predictions inside the training regions, used as a tuning diagnostic;
- one final fit on all training regions and one prediction on the fixed `PT16E` + `PT16J` test set, used for performance analysis.


In [4]:
SPATIAL_CV_FOLDS = 5
SEARCH_CANDIDATES = 40

preprocess = ColumnTransformer(
    transformers=[("num", SimpleImputer(strategy="median"), predictor_cols)],
    remainder="drop",
    verbose_feature_names_out=False,
)
pipe = Pipeline([
    ("prep", preprocess),
    ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, bootstrap=True)),
])

# Broad random search for the first region-held-out tuning pass.
param_distributions = {
    "model__n_estimators": [300, 500, 800],
    "model__max_features": ["sqrt", 0.35, 0.5, 0.75, 1.0],
    "model__min_samples_leaf": [1, 2, 4, 8, 12],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__max_depth": [None, 8, 12, 18, 24, 32],
}

candidate_space = len(ParameterGrid(param_distributions))
print("Spatial CV folds:", SPATIAL_CV_FOLDS)
print("Random search candidates per target:", SEARCH_CANDIDATES)
print("Broad candidate space size:", candidate_space)
print("Maximum search fits per target:", SEARCH_CANDIDATES * SPATIAL_CV_FOLDS)


Spatial CV folds: 5
Random search candidates per target: 40
Broad candidate space size: 1800
Maximum search fits per target: 200


In [5]:
def fmt_seconds(seconds: float) -> str:
    seconds = int(round(seconds))
    hours, rem = divmod(seconds, 3600)
    minutes, secs = divmod(rem, 60)
    if hours:
        return f"{hours}h {minutes}m {secs}s"
    if minutes:
        return f"{minutes}m {secs}s"
    return f"{secs}s"


def metric_row(prefix: str, y_true, y_pred) -> dict[str, float]:
    return {
        f"{prefix}_r2": r2_score(y_true, y_pred),
        f"{prefix}_mae": mean_absolute_error(y_true, y_pred),
        f"{prefix}_rmse": rmse(y_true, y_pred),
        f"{prefix}_spearman": spearman_corr(y_true, y_pred),
    }

metrics = []
prediction_frames = []
importance_frames = []
fold_frames = []
best_params = {}
all_started = time.perf_counter()

for target_idx, target in enumerate(EPVI_TARGETS, start=1):
    target_started = time.perf_counter()
    train_t = train_df.dropna(subset=[target]).copy().reset_index(drop=True)
    test_t = test_df.dropna(subset=[target]).copy().reset_index(drop=True)
    spatial_cv, fold_summary = make_nuts3_cv_splits(train_t, n_splits=SPATIAL_CV_FOLDS)
    fold_summary.insert(0, "target", target)
    fold_frames.append(fold_summary)

    X_train = train_t[predictor_cols]
    y_train = train_t[target].astype(float)
    X_test = test_t[predictor_cols]
    y_test = test_t[target].astype(float)

    print("=" * 96, flush=True)
    print(f"[{target_idx}/{len(EPVI_TARGETS)}] Target: {target}", flush=True)
    print(
        f"Train rows={len(train_t):,} | test rows={len(test_t):,} | predictors={len(predictor_cols):,} | "
        f"NUTS3 CV folds={len(spatial_cv)} | search fits={SEARCH_CANDIDATES * len(spatial_cv):,}",
        flush=True,
    )
    display(fold_summary)

    search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=param_distributions,
        n_iter=SEARCH_CANDIDATES,
        scoring="r2",
        cv=spatial_cv,
        n_jobs=-1,
        refit=True,
        verbose=2,
        random_state=RANDOM_STATE,
    )
    print(f"Starting spatial hyperparameter search at {datetime.now().strftime('%H:%M:%S')}...", flush=True)
    search.fit(X_train, y_train)
    best = search.best_estimator_
    best_params[target] = search.best_params_
    print(f"Best mean spatial-CV search R2={search.best_score_:.4f}", flush=True)
    print("Best params:", best_params[target], flush=True)

    print("Computing spatial out-of-fold predictions on training regions...", flush=True)
    cv_pred = cross_val_predict(best, X_train, y_train, cv=spatial_cv, n_jobs=-1)
    print("Refitting selected model on all training regions and predicting fixed test regions...", flush=True)
    best.fit(X_train, y_train)
    train_pred = best.predict(X_train)
    test_pred = best.predict(X_test)

    row = {
        "target": target,
        "n_train_rows": len(train_t),
        "n_test_rows": len(test_t),
        "n_predictors": len(predictor_cols),
        "search_spatial_cv_r2_mean": search.best_score_,
        **metric_row("train_spatial_cv", y_train, cv_pred),
        **metric_row("fixed_test", y_test, test_pred),
        **metric_row("train_in_sample", y_train, train_pred),
    }
    metrics.append(row)

    for role, data, observed, predicted in [
        ("train_spatial_cv", train_t, y_train, cv_pred),
        ("fixed_test", test_t, y_test, test_pred),
        ("train_in_sample", train_t, y_train, train_pred),
    ]:
        prediction_frames.append(pd.DataFrame({
            "prediction_role": role,
            "target": target,
            "ID": data["ID"].values,
            "ID_norm": data["ID_norm"].values,
            "NUTS3": data["NUTS3"].values,
            "name": data["parish_name"].values,
            "observed": observed.values,
            "predicted": predicted,
            "residual": observed.values - predicted,
        }))

    importance_frames.append(pd.DataFrame({
        "target": target,
        "feature": predictor_cols,
        "impurity_importance": best.named_steps["model"].feature_importances_,
    }).sort_values("impurity_importance", ascending=False))

    target_seconds = time.perf_counter() - target_started
    elapsed_total = time.perf_counter() - all_started
    remaining = elapsed_total / target_idx * (len(EPVI_TARGETS) - target_idx)
    print(pd.Series(row).to_string(), flush=True)
    print(f"Finished in {fmt_seconds(target_seconds)}. Elapsed={fmt_seconds(elapsed_total)} | ETA={fmt_seconds(remaining)}", flush=True)

metrics_df = pd.DataFrame(metrics).sort_values("fixed_test_r2", ascending=False)
predictions_df = pd.concat(prediction_frames, ignore_index=True)
importances_df = pd.concat(importance_frames, ignore_index=True)
folds_df = pd.concat(fold_frames, ignore_index=True)
display(metrics_df)


[1/5] Target: EPG heating
Train rows=2,653 | test rows=434 | predictors=46 | NUTS3 CV folds=5 | search fits=200


,target,fold,n_rows,n_nuts3,validation_nuts3
0,EPG heating,0,513,4,"PT11A, PT150, PT170, PT200"
1,EPG heating,1,569,6,"PT11B, PT11C, PT16H, PT16I, PT187, PT300"
2,EPG heating,2,514,4,"PT11D, PT16D, PT16F, PT16G"
3,EPG heating,3,542,5,"PT111, PT119, PT181, PT185, PT186"
4,EPG heating,4,515,4,"PT112, PT11E, PT16B, PT184"


Starting spatial hyperparameter search at 21:53:39...
Fitting 5 folds for each of 40 candidates, totalling 200 fits
Best mean spatial-CV search R2=0.0251
Best params: {'model__n_estimators': 500, 'model__min_samples_split': 20, 'model__min_samples_leaf': 2, 'model__max_features': 'sqrt', 'model__max_depth': 8}
Computing spatial out-of-fold predictions on training regions...
Refitting selected model on all training regions and predicting fixed test regions...
target                       EPG heating
n_train_rows                        2653
n_test_rows                          434
n_predictors                          46
search_spatial_cv_r2_mean       0.025079
train_spatial_cv_r2             0.218791
train_spatial_cv_mae            1.014254
train_spatial_cv_rmse           1.545589
train_spatial_cv_spearman       0.390987
fixed_test_r2                    0.05266
fixed_test_mae                  1.050453
fixed_test_rmse                 1.690758
fixed_test_spearman             0.265448
trai

,target,fold,n_rows,n_nuts3,validation_nuts3
0,EPG cooling,0,513,4,"PT11A, PT150, PT170, PT200"
1,EPG cooling,1,569,6,"PT11B, PT11C, PT16H, PT16I, PT187, PT300"
2,EPG cooling,2,514,4,"PT11D, PT16D, PT16F, PT16G"
3,EPG cooling,3,542,5,"PT111, PT119, PT181, PT185, PT186"
4,EPG cooling,4,515,4,"PT112, PT11E, PT16B, PT184"


Starting spatial hyperparameter search at 22:01:41...
Fitting 5 folds for each of 40 candidates, totalling 200 fits
Best mean spatial-CV search R2=-0.0573
Best params: {'model__n_estimators': 500, 'model__min_samples_split': 20, 'model__min_samples_leaf': 2, 'model__max_features': 'sqrt', 'model__max_depth': 8}
Computing spatial out-of-fold predictions on training regions...
Refitting selected model on all training regions and predicting fixed test regions...
target                       EPG cooling
n_train_rows                        2653
n_test_rows                          434
n_predictors                          46
search_spatial_cv_r2_mean      -0.057347
train_spatial_cv_r2             0.131915
train_spatial_cv_mae            0.720884
train_spatial_cv_rmse           0.993621
train_spatial_cv_spearman       0.395818
fixed_test_r2                  -0.673809
fixed_test_mae                  0.837574
fixed_test_rmse                 0.931969
fixed_test_spearman             0.259734
tra

...

,target,fold,n_rows,n_nuts3,validation_nuts3
0,EPVI heating,0,513,4,"PT11A, PT150, PT170, PT200"
1,EPVI heating,1,569,6,"PT11B, PT11C, PT16H, PT16I, PT187, PT300"
2,EPVI heating,2,514,4,"PT11D, PT16D, PT16F, PT16G"
3,EPVI heating,3,542,5,"PT111, PT119, PT181, PT185, PT186"
4,EPVI heating,4,515,4,"PT112, PT11E, PT16B, PT184"


Starting spatial hyperparameter search at 22:19:06...
Fitting 5 folds for each of 40 candidates, totalling 200 fits
Best mean spatial-CV search R2=0.3916
Best params: {'model__n_estimators': 300, 'model__min_samples_split': 5, 'model__min_samples_leaf': 2, 'model__max_features': 0.75, 'model__max_depth': 8}
Computing spatial out-of-fold predictions on training regions...
Refitting selected model on all training regions and predicting fixed test regions...
target                       EPVI heating
n_train_rows                         2653
n_test_rows                           434
n_predictors                           46
search_spatial_cv_r2_mean        0.391564
train_spatial_cv_r2              0.449953
train_spatial_cv_mae             0.555285
train_spatial_cv_rmse            0.838075
train_spatial_cv_spearman        0.712442
fixed_test_r2                    0.318802
fixed_test_mae                   0.573736
fixed_test_rmse                  0.897745
fixed_test_spearman              0.6

,target,fold,n_rows,n_nuts3,validation_nuts3
0,EPVI cooling,0,513,4,"PT11A, PT150, PT170, PT200"
1,EPVI cooling,1,569,6,"PT11B, PT11C, PT16H, PT16I, PT187, PT300"
2,EPVI cooling,2,514,4,"PT11D, PT16D, PT16F, PT16G"
3,EPVI cooling,3,542,5,"PT111, PT119, PT181, PT185, PT186"
4,EPVI cooling,4,515,4,"PT112, PT11E, PT16B, PT184"


Starting spatial hyperparameter search at 22:27:47...
Fitting 5 folds for each of 40 candidates, totalling 200 fits
Best mean spatial-CV search R2=0.3633
Best params: {'model__n_estimators': 500, 'model__min_samples_split': 10, 'model__min_samples_leaf': 1, 'model__max_features': 1.0, 'model__max_depth': None}
Computing spatial out-of-fold predictions on training regions...
Refitting selected model on all training regions and predicting fixed test regions...
target                       EPVI cooling
n_train_rows                         2653
n_test_rows                           434
n_predictors                           46
search_spatial_cv_r2_mean          0.3633
train_spatial_cv_r2              0.478367
train_spatial_cv_mae             0.419545
train_spatial_cv_rmse            0.565592
train_spatial_cv_spearman        0.722596
fixed_test_r2                    0.371941
fixed_test_mae                   0.435124
fixed_test_rmse                  0.505674
fixed_test_spearman              

,target,n_train_rows,n_test_rows,n_predictors,search_spatial_cv_r2_mean,train_spatial_cv_r2,train_spatial_cv_mae,train_spatial_cv_rmse,train_spatial_cv_spearman,fixed_test_r2,fixed_test_mae,fixed_test_rmse,fixed_test_spearman,train_in_sample_r2,train_in_sample_mae,train_in_sample_rmse,train_in_sample_spearman
2,AIAM,2653,434,46,0.849133,0.855464,0.261404,0.380651,0.928522,0.909171,0.234074,0.322056,0.952655,0.973927,0.112890,0.161673,0.987096
4,EPVI cooling,2653,434,46,0.363300,0.478367,0.419545,0.565592,0.722596,0.371941,0.435124,0.505674,0.740673,0.936946,0.142474,0.196642,0.970086
3,EPVI heating,2653,434,46,0.391564,0.449953,0.555285,0.838075,0.712442,0.318802,0.573736,0.897745,0.664359,0.839016,0.337278,0.453392,0.879469
0,EPG heating,2653,434,46,0.025079,0.218791,1.014254,1.545589,0.390987,0.052660,1.050453,1.690758,0.265448,0.652386,0.712536,1.031002,0.733660
1,EPG cooling,2653,434,46,-0.057347,0.131915,0.720884,0.993621,0.395818,-0.673809,0.837574,0.931969,0.259734,0.700121,0.408571,0.584000,0.859038


In [6]:
# Optional feature importance diagnostic evaluated on the fixed test set.
RUN_PERMUTATION_IMPORTANCE = False
permutation_frames = []
if RUN_PERMUTATION_IMPORTANCE:
    for target in EPVI_TARGETS:
        train_t = train_df.dropna(subset=[target]).copy()
        test_t = test_df.dropna(subset=[target]).copy()
        best = Pipeline([
            ("prep", preprocess),
            ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, bootstrap=True, **{
                key.replace("model__", ""): value for key, value in best_params[target].items()
            })),
        ])
        best.fit(train_t[predictor_cols], train_t[target].astype(float))
        perm = permutation_importance(
            best,
            test_t[predictor_cols],
            test_t[target].astype(float),
            scoring="r2",
            n_repeats=10,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        permutation_frames.append(pd.DataFrame({
            "target": target,
            "feature": predictor_cols,
            "permutation_importance_mean": perm.importances_mean,
            "permutation_importance_std": perm.importances_std,
        }).sort_values("permutation_importance_mean", ascending=False))
permutation_df = pd.concat(permutation_frames, ignore_index=True) if permutation_frames else pd.DataFrame()
if not permutation_df.empty:
    display(permutation_df.groupby("target").head(15))


In [7]:
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
metrics_path = MODEL_OUT_DIR / f"rf_epvi_metrics_{stamp}.csv"
predictions_path = MODEL_OUT_DIR / f"rf_epvi_predictions_residuals_{stamp}.csv"
importances_path = MODEL_OUT_DIR / f"rf_epvi_feature_importances_{stamp}.csv"
params_path = MODEL_OUT_DIR / f"rf_epvi_best_params_{stamp}.json"
folds_path = MODEL_OUT_DIR / f"rf_epvi_spatial_cv_folds_{stamp}.csv"

metrics_df.to_csv(metrics_path, index=False)
predictions_df.to_csv(predictions_path, index=False)
importances_df.to_csv(importances_path, index=False)
folds_df.to_csv(folds_path, index=False)
with open(params_path, "w", encoding="utf-8") as f:
    json.dump(best_params, f, indent=2, ensure_ascii=False)

if not permutation_df.empty:
    permutation_path = MODEL_OUT_DIR / f"rf_epvi_permutation_importances_{stamp}.csv"
    permutation_df.to_csv(permutation_path, index=False)
else:
    permutation_path = None

print("Wrote:")
for path in [metrics_path, predictions_path, importances_path, params_path, folds_path, permutation_path]:
    if path is not None:
        print(" -", path)


Wrote:
 - E:\OneDrive\Studia\Studia magisterskie\Masterarbeit 2 - Sozialwissenschaften\data\outputs\epvi_prediction\random_forest\rf_epvi_metrics_20260521_223809.csv
 - E:\OneDrive\Studia\Studia magisterskie\Masterarbeit 2 - Sozialwissenschaften\data\outputs\epvi_prediction\random_forest\rf_epvi_predictions_residuals_20260521_223809.csv
 - E:\OneDrive\Studia\Studia magisterskie\Masterarbeit 2 - Sozialwissenschaften\data\outputs\epvi_prediction\random_forest\rf_epvi_feature_importances_20260521_223809.csv
 - E:\OneDrive\Studia\Studia magisterskie\Masterarbeit 2 - Sozialwissenschaften\data\outputs\epvi_prediction\random_forest\rf_epvi_best_params_20260521_223809.json
 - E:\OneDrive\Studia\Studia magisterskie\Masterarbeit 2 - Sozialwissenschaften\data\outputs\epvi_prediction\random_forest\rf_epvi_spatial_cv_folds_20260521_223809.csv


## Next Step

Use the fixed-test metrics for model performance reporting. Use the training-region spatial-CV metrics to decide whether another narrower tuning pass is warranted, and inspect fixed-test residuals before moving detailed administrative indicators into the residual-explanation stage.
